# **6일차 과제 — 진료 데이터 기반 QA 챗봇 구현하기2 (답변품질 검증)**

* 5일차 QA 챗봇에 **CRAG**(검색 문서 관련성 확인 + 웹 검색 보완)와 **Self-RAG**(근거 확인 `is_grounded`, 유용성 확인 `is_useful`) 품질 검증을 추가합니다.
* 동일한 질문에 대해 **검증 없는 기본 RAG 챗봇**과 **검증이 추가된 챗봇**의 답변을 비교합니다.
* ⚠️ 4일차에서 생성된 `chronic_db` 폴더를 이번 노트북과 같은 위치에 두고 실행하세요.

> **제출자: 조태희** · OZ코딩스쿨 AI 헬스케어 부트캠프 6기 · 자연어 처리(NLP) 6일차 과제 (RAG 기초3 · RAG 확장)

**📌 실행 안내**
- 실행 번호와 출력이 있는 셀은 API 키·벡터DB 없이도 돌아가는 셀(State·노드 함수 정의, 분기 함수 테스트, 두 챗봇의 그래프 조립·구조 확인)이라 미리 실행해 둔 결과입니다.
- 코드 첫 줄에 **🔑 [Colab에서 실행]** 표시가 있는 셀은 **OpenAI·Tavily API 키**와 4일차 벡터DB(`chronic_db`)가 필요해서 아직 실행하지 않았습니다.
- Colab 준비물: `api_key.txt`(두 키가 한 줄씩) 업로드 또는 🔑 보안 비밀 등록, 그리고 `chronic_db` 폴더. 4일차에서 드라이브 `MyDrive/ai_agent/`에 저장해 두었다면 **(3) 벡터DB 연결**의 추가 셀이 노트북 위치로 복사해 옵니다. 준비되면 **런타임 → 모두 실행**으로 위에서부터 차례대로 실행해 주세요.
- GPT 답변과 웹 검색 결과는 실행할 때마다 조금씩 달라질 수 있습니다.

# **1. 환경준비**

## **(1) 라이브러리**

In [ ]:
# 🔑 [Colab에서 실행] 필요한 라이브러리 설치 (-q: 설치 로그를 짧게 표시)
!pip install langchain-openai langchain-tavily langgraph langchain-chroma -q

In [1]:
import os   # 환경변수(API 키), 파일 경로 관리

from langchain_openai import ChatOpenAI, OpenAIEmbeddings               # GPT 모델 / 임베딩 모델
from langchain_core.messages import HumanMessage, SystemMessage         # 메시지 종류
from langchain_core.documents import Document                           # 문서 객체 (웹 검색 결과를 DB 문서와 같은 형태로 담을 때 사용)
from langchain_tavily import TavilySearch                               # Tavily 웹 검색 도구
from langchain_chroma import Chroma                                     # 벡터DB(Chroma)
from langgraph.graph import StateGraph, START, END, MessagesState       # 그래프 도구 + messages 칸이 있는 기본 State

## **(2) API Key 확인**

In [ ]:
# 🔑 [Colab에서 실행] api_key.txt 파일 업로드
#   - api_key.txt 안에는 아래처럼 한 줄에 하나씩 적혀 있어야 합니다.
#       OPENAI_API_KEY=sk-...
#       TAVILY_API_KEY=tvly-...
#   - Colab 왼쪽 🔑(보안 비밀)에 두 키를 등록해 두었다면 이 셀은 건너뛰어도 됩니다.
from google.colab import files   # Colab에서 내 컴퓨터 파일을 올리는 기능
uploaded = files.upload()        # 업로드 창이 뜨면 api_key.txt를 선택합니다

In [ ]:
# 🔑 [Colab에서 실행] API 키를 환경변수로 등록합니다 (코드에 키를 직접 적지 않기 위해서예요)
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:                      # 키 파일을 읽기 모드로 엽니다
        for line in f:                                  # 한 줄씩 읽으면서
            line = line.strip()                         # 앞뒤 공백·줄바꿈을 지우고
            if line and "=" in line:                    # 빈 줄이 아니고 "이름=값" 형태라면
                key, value = line.split("=", 1)         # 첫 번째 '='를 기준으로 이름과 값을 나눠서
                os.environ[key.strip()] = value.strip() # 환경변수로 등록합니다

def ensure_api_key(name):
    """환경변수에 키가 없으면 Colab 보안 비밀 → 직접 입력 순서로 채웁니다."""
    if os.environ.get(name):                            # 이미 등록돼 있으면 그대로 사용
        return                                          # 더 할 일이 없으니 함수 종료
    try:
        from google.colab import userdata               # Colab 🔑 보안 비밀 저장소
        os.environ[name] = userdata.get(name)           # 보안 비밀에 등록된 키를 꺼내 등록
    except Exception:                                   # 보안 비밀에 없거나 Colab이 아니면
        from getpass import getpass                     # 입력한 글자가 화면에 안 보이는 입력창
        os.environ[name] = getpass(f"{name}를 입력하세요: ")   # 직접 입력받아 등록


if os.path.exists("api_key.txt"):                       # 업로드한 키 파일이 있으면
    load_api_keys("api_key.txt")                        # 파일에서 읽어서 등록
ensure_api_key("OPENAI_API_KEY")                        # OpenAI 키 (LLM + 임베딩)
ensure_api_key("TAVILY_API_KEY")                        # Tavily 키 (CRAG 웹 검색 보완)

* 아래 코드셀은, 실행해서 key가 제대로 보이는지 확인하고 삭제하세요.

In [ ]:
# 🔑 [Colab에서 실행] 키가 잘 들어왔는지 앞부분만 확인합니다
#   (전체 키가 노트북에 남으면 유출될 수 있어서 앞 7글자만 보이게 했습니다. 확인 후 이 셀은 삭제하세요.)
print(os.environ['OPENAI_API_KEY'][:7] + "...")
print(os.environ['TAVILY_API_KEY'][:7] + "...")

## **(3) 벡터DB 연결**
* 4일차에서 구축한 `chronic_db`를 그대로 불러옵니다.

In [ ]:
# 🔑 [Colab에서 실행] (추가) chronic_db가 노트북 위치(/content)에 없으면 구글 드라이브에서 복사해 옵니다
#   4일차 마지막 셀로 드라이브 MyDrive/ai_agent/chronic_db 에 저장해 둔 경우에 사용하세요.
import shutil                                            # 폴더 복사 도구
if not os.path.exists("./chronic_db"):                   # 노트북 위치에 DB 폴더가 없으면
    from google.colab import drive                       # 구글 드라이브 연결 도구
    drive.mount('/content/drive')                        # 드라이브 연결
    src = '/content/drive/MyDrive/ai_agent/chronic_db'   # 4일차에서 저장해 둔 위치
    if os.path.exists(src):                              # 드라이브에 DB가 있으면
        shutil.copytree(src, './chronic_db')             # 드라이브 → 노트북 위치로 복사
    else:                                                # 없으면 어떻게 해야 하는지 안내
        print("⚠️ 드라이브에서 chronic_db를 찾지 못했습니다. 4일차 노트북 마지막 셀로 저장하거나, "
              "chronic_db 폴더를 /content에 직접 올려 주세요.")
print("chronic_db 준비:", os.path.exists("./chronic_db"))   # True면 준비 완료

In [ ]:
# 🔑 [Colab에서 실행] 벡터DB·LLM·웹 검색 도구 준비 (객체를 만들 때 API 키를 확인합니다)
embedding = OpenAIEmbeddings(model="text-embedding-3-small")   # ⚠️ 4일차에서 DB를 만들 때와 같은 임베딩 모델

vectorstore = Chroma(embedding_function=embedding,
                      persist_directory="./chronic_db")         # 노트북 위치의 chronic_db 연결

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)           # 답변 생성·평가에 쓸 GPT
web_search_tool = TavilySearch(max_results=3)                   # CRAG 웹 검색 보완용 (결과 3개)

print("불러온 청크 수:", vectorstore._collection.count())        # 0이면 chronic_db 위치를 확인하세요

## **(4) State 정의**

In [2]:
class QAState(MessagesState):
    question: str      # 사용자 질문
    documents: list     # 검색된 문서
    grade: str          # 문서 관련성 평가: 'yes' or 'no'
    answer: str          # 생성된 답변
    grounded: str        # 근거 확인 결과: 'yes' or 'no'
    useful: str          # 유용성 확인 결과: 'yes' or 'no'
    retry_count: int     # (추가) 근거 부족으로 답변을 다시 쓴 횟수 — 무한 반복을 막는 탈출 조건에 사용

# **2. 기본 RAG 챗봇 (검증 없음)**

* 검색된 문서가 질문과 관련이 있는지, 답변이 문서에 근거하는지 확인하지 않고 바로 답변하는 챗봇입니다.
* `retrieve` → `generate` 2노드 구조 (이 두 노드는 3절 CRAG/Self-RAG 챗봇에서도 그대로 재사용됩니다).

**💡 풀이 포인트** — 기본 RAG는 "검색(`retrieve`) → 답변(`generate`)" 일직선입니다. 생성 지침은 일부러 단순하게("참고 문서를 바탕으로 구체적으로 답하라") 두었는데, 실무에서 흔한 형태이면서 3절의 검증 단계가 **무엇을 막아 주는지** 비교하기 좋기 때문입니다. 이 두 노드는 3절에서 그대로 재사용합니다.

In [3]:
# [기본 RAG ①] retrieve 노드 — 벡터DB에서 질문과 비슷한 문서 3개를 찾아 State에 저장
def retrieve(state: QAState):
    question = state["messages"][-1].content                   # 가장 최근 사용자 메시지 = 질문
    documents = vectorstore.similarity_search(question, k=3)   # Document 객체 3개 (내용 + 메타데이터)
    return {"question": question, "documents": documents}      # question, documents 칸 채우기

In [4]:
# [기본 RAG ②] generate 노드 — 문서를 참고해 답변 생성 (관련성·근거 검증 없음)
def generate(state: QAState):
    context = "\n\n---\n\n".join(doc.page_content for doc in state["documents"])   # 문서 내용을 구분선으로 이어 붙이기
    s_msg = SystemMessage(content="당신은 만성질환 환자를 돕는 병원 QA 챗봇입니다. "
                                  "아래 [참고 문서]를 바탕으로 사용자의 질문에 친절하고 구체적으로 답변하세요.")
    h_msg = HumanMessage(content=f"[참고 문서]\n{context}\n\n[질문]\n{state['question']}")   # 문서 + 질문
    response = llm.invoke([s_msg, h_msg])                      # 답변 생성
    return {"messages": [response], "answer": response.content}

In [5]:
builder = StateGraph(QAState)              # 질문·문서·평가 칸이 있는 State
builder.add_node("retrieve", retrieve)     # 검색 노드
builder.add_node("generate", generate)     # 답변 노드
builder.add_edge(START, "retrieve")        # 시작 → 검색
builder.add_edge("retrieve", "generate")   # 검색 → 답변 (검증 없이 바로)
builder.add_edge("generate", END)          # 답변 → 끝
basic_graph = builder.compile()            # 완성

# 그래프 연결 구조 확인 (__start__ = START, __end__ = END)
drawn = basic_graph.get_graph()                                              # 그래프 구조(노드·엣지) 정보 꺼내기
node_order = list(drawn.nodes)                                       # 노드를 등록한 순서
for edge in sorted(drawn.edges, key=lambda e: node_order.index(e.source)):   # 출발 노드 순서대로 정렬해서 하나씩
    label = ""                                                       # 기본값: 조건 표시 없음
    if edge.conditional:                                             # 조건부 연결이면 어떤 조건인지 표시
        label = f"   (조건부: {edge.data})" if edge.data else "   (조건부)"
    print(edge.source, "→", edge.target, label)

__start__ → retrieve 
retrieve → generate 
generate → __end__ 


In [ ]:
# 🔑 [Colab에서 실행] 기본 RAG 챗봇 실행 (API 키 + 벡터DB 필요)
basic_result = basic_graph.invoke({"messages": [HumanMessage(content="고혈압 환자가 주의해야 할 음식은?")]})   # 질문을 넣어 실행

print("[검색된 문서 수]", len(basic_result["documents"]))
print("[답변]\n" + basic_result["answer"])

# **3. CRAG + Self-RAG 적용 (검증 있음)**

**💡 풀이 포인트** — **CRAG**(Corrective RAG, 교정형 RAG)는 **답을 만들기 전에** 재료(검색 문서)가 질문과 관련 있는지 LLM으로 하나씩 채점하고, 관련 문서가 하나도 없으면 웹 검색으로 재료를 보충합니다. **Self-RAG**(자기 점검 RAG)는 **답을 만든 후에** 그 답이 문서에 근거하는지(`is_grounded`)와 질문 해결에 도움이 되는지(`is_useful`)를 스스로 점검합니다. 근거가 부족하면 문서 내용만으로 한 번 다시 쓰고(`regenerate`), 그래도 안 되면 지어내는 대신 "근거를 찾지 못했다"고 안내(`fallback`)하도록 했습니다. 무한 반복을 막으려고 State에 `retry_count`(다시 쓴 횟수) 칸을 하나 추가했어요.

## **(1) CRAG — 문서 관련성 평가 및 웹 검색 보완**

In [6]:
# [CRAG ①] grade_documents 노드 — 검색 문서를 하나씩 채점해서 관련 있는 문서만 남깁니다
def grade_documents(state: QAState):
    s_msg = SystemMessage(content="당신은 검색된 문서가 사용자 질문과 관련이 있는지 평가하는 평가자입니다. "
                                  "문서에 질문에 답하는 데 도움이 되는 키워드나 의미가 담겨 있으면 'yes', "
                                  "관련이 없으면 'no'라고만 답하세요. 다른 말은 붙이지 마세요.")
    relevant_docs = []                                         # 관련 있다고 판정된 문서만 모을 리스트
    for doc in state["documents"]:                             # 검색된 문서를 하나씩 평가
        h_msg = HumanMessage(content=f"[질문]\n{state['question']}\n\n[문서]\n{doc.page_content}")   # 질문 + 문서 한 개
        result = llm.invoke([s_msg, h_msg])                    # LLM 채점
        if "yes" in result.content.lower():                    # 'yes'(대소문자 무관)면 남기고, 'no'면 버림
            relevant_docs.append(doc)                          # 관련 문서 보관
    grade = "yes" if relevant_docs else "no"                   # 관련 문서가 하나라도 있으면 'yes'
    return {"documents": relevant_docs, "grade": grade}        # 관련 문서만 남기고 평가 결과 저장

In [7]:
# [CRAG ②] web_search 노드 — 관련 문서가 없을 때 Tavily 웹 검색 결과로 재료를 보충합니다
def web_search(state: QAState):
    try:
        response = web_search_tool.invoke({"query": state["question"]})        # 질문 그대로 웹 검색
        results = response.get("results", []) if isinstance(response, dict) else []   # 결과가 0건이면 안내 문구(문자열)가 와서 빈 리스트로 처리
        web_docs = [Document(page_content=r["content"],                       # 검색 결과 → Document 형태로 변환
                             metadata={"source": r.get("url", "")})           # 출처(URL)는 메타데이터에 보관
                    for r in results if r.get("content")]                     # 내용이 있는 결과만
    except Exception as e:                                                    # 검색 실패 시 빈 결과로 진행
        print("웹 검색 실패:", e)
        web_docs = []                                                         # 웹 문서 없음
    return {"documents": state["documents"] + web_docs}                       # (남은 DB 문서) + 웹 문서

## **(2) Self-RAG — 근거 확인(is_grounded) / 유용성 확인(is_useful)**

In [8]:
# [Self-RAG ①] is_grounded 노드 — 답변 내용이 참고 문서에서 확인되는지 검사
def is_grounded(state: QAState):
    context = "\n\n---\n\n".join(doc.page_content for doc in state["documents"])   # 검사 기준이 될 문서 내용
    s_msg = SystemMessage(content="당신은 답변이 참고 문서에 근거하는지 확인하는 검증자입니다. "
                                  "답변의 핵심 내용이 모두 [참고 문서]에서 확인되면 'yes', "
                                  "문서에 없는 내용(지어낸 사실, 특정 제품명·수치 등)이 들어 있으면 'no'라고만 답하세요. "
                                  "답변이 '문서에서 확인할 수 없다'고 솔직하게 밝히는 부분은 지어낸 내용이 아니므로 근거 부족으로 보지 마세요.")
    h_msg = HumanMessage(content=f"[참고 문서]\n{context}\n\n[답변]\n{state['answer']}")   # 문서와 답변을 나란히 보여 줌
    result = llm.invoke([s_msg, h_msg])                                   # LLM 검증
    grounded = "yes" if "yes" in result.content.lower() else "no"         # yes/no 두 값으로 정리
    return {"grounded": grounded}


# [Self-RAG ①-보완] regenerate 노드 — 근거가 부족할 때 '문서에 있는 내용만'으로 다시 쓰기
def regenerate(state: QAState):
    context = "\n\n---\n\n".join(doc.page_content for doc in state["documents"])   # 다시 쓸 때 쓸 수 있는 유일한 재료
    s_msg = SystemMessage(content="당신은 만성질환 환자를 돕는 병원 QA 챗봇입니다. 이전 답변에 참고 문서에 없는 내용이 섞여 있었습니다. "
                                  "이번에는 [참고 문서]에 적힌 내용만 사용해서 다시 답변하세요. "
                                  "문서에서 확인할 수 없는 내용(특정 제품명, 수치 등)은 절대 지어내지 말고 "
                                  "'제공된 자료에서는 확인할 수 없습니다'라고 분명히 밝히세요.")
    h_msg = HumanMessage(content=f"[참고 문서]\n{context}\n\n[질문]\n{state['question']}\n\n"
                                 f"[이전 답변(근거 부족)]\n{state['answer']}")
    response = llm.invoke([s_msg, h_msg])                                 # 다시 쓰기
    return {"messages": [response],
            "answer": response.content,                                   # 새 답변으로 교체
            "retry_count": state.get("retry_count", 0) + 1}               # 다시 쓴 횟수 +1

In [9]:
from langchain_core.messages import AIMessage   # fallback 노드에서 직접 만든 안내 문구를 AI 메시지로 담기 위해


# [Self-RAG ②] is_useful 노드 — 답변이 질문을 실제로 해결하는지 검사
def is_useful(state: QAState):
    s_msg = SystemMessage(content="당신은 답변이 사용자 질문을 실제로 해결하는지 평가하는 평가자입니다. "
                                  "답변이 질문에 직접적으로 도움이 되는 정보를 주면 'yes', "
                                  "질문과 동떨어졌거나 아무 도움이 되지 않으면 'no'라고만 답하세요.")
    h_msg = HumanMessage(content=f"[질문]\n{state['question']}\n\n[답변]\n{state['answer']}")   # 질문과 답변을 나란히 보여 줌
    result = llm.invoke([s_msg, h_msg])                                   # LLM 평가
    useful = "yes" if "yes" in result.content.lower() else "no"           # yes/no 두 값으로 정리
    return {"useful": useful}


# [Self-RAG ②-보완] fallback 노드 — 끝내 검증을 통과하지 못하면 지어내지 않고 솔직하게 안내
def fallback(state: QAState):
    # 사용자에게 보여 줄 안내 문구 (근거 없이 지어내지 않고 솔직하게 안내)
    message = ("죄송합니다. 제공된 자료와 검색 결과에서 이 질문에 대한 믿을 만한 근거를 찾지 못해 정확히 답변드리기 어렵습니다. "
               "정확한 내용은 담당 의료진이나 공식 기관에 확인해 주세요.")
    return {"messages": [AIMessage(content=message)], "answer": message}  # 최종 답변을 안내 문구로 교체

## **(3) 조건 분기 함수 및 그래프 구성**

In [10]:
def decide_after_grade(state: QAState) -> str:
    # CRAG: 관련 문서가 있으면 바로 답변, 하나도 없으면 웹 검색으로 보완
    return "generate" if state["grade"] == "yes" else "web_search"


def decide_after_grounded(state: QAState) -> str:
    # Self-RAG ①: 근거가 확인되면 유용성 검사로 / 아니면 1번까지 다시 쓰고, 그래도 안 되면 포기(fallback)
    if state["grounded"] == "yes":
        return "grounded"
    if state.get("retry_count", 0) < 1:
        return "retry"
    return "give_up"


def decide_after_useful(state: QAState) -> str:
    # Self-RAG ②: 질문 해결에 도움이 되면 종료 / 아니면 fallback
    return "useful" if state["useful"] == "yes" else "not_useful"


# 분기 함수만 따로 테스트 (LLM 없이, 직접 만든 State로 어느 길로 가는지 확인)
print("grade=yes                  →", decide_after_grade({"grade": "yes"}))
print("grade=no                   →", decide_after_grade({"grade": "no"}))
print("grounded=yes               →", decide_after_grounded({"grounded": "yes", "retry_count": 0}))
print("grounded=no, retry_count=0 →", decide_after_grounded({"grounded": "no", "retry_count": 0}))
print("grounded=no, retry_count=1 →", decide_after_grounded({"grounded": "no", "retry_count": 1}))
print("useful=no                  →", decide_after_useful({"useful": "no"}))

grade=yes                  → generate
grade=no                   → web_search
grounded=yes               → grounded
grounded=no, retry_count=0 → retry
grounded=no, retry_count=1 → give_up
useful=no                  → not_useful


In [11]:
builder = StateGraph(QAState)                          # 질문·문서·평가 칸이 있는 State로 설계도 만들기
builder.add_node("retrieve", retrieve)                 # 1) 벡터DB 검색 (2절 노드 재사용)
builder.add_node("grade_documents", grade_documents)   # 2) CRAG: 문서 관련성 평가
builder.add_node("web_search", web_search)             # 3) CRAG: 관련 문서가 없으면 웹 검색 보완
builder.add_node("generate", generate)                 # 4) 답변 생성 (2절 노드 재사용)
builder.add_node("is_grounded", is_grounded)           # 5) Self-RAG: 근거 확인
builder.add_node("regenerate", regenerate)             # 5-1) 근거 부족 시 문서 내용만으로 다시 쓰기
builder.add_node("is_useful", is_useful)               # 6) Self-RAG: 유용성 확인
builder.add_node("fallback", fallback)                 # 7) 끝내 검증 실패 시 안전한 안내 문구

builder.add_edge(START, "retrieve")                    # 시작 → 검색
builder.add_edge("retrieve", "grade_documents")        # 검색 → 관련성 평가
builder.add_conditional_edges("grade_documents", decide_after_grade,   # 관련성 평가 후 갈림길
                              {"generate": "generate", "web_search": "web_search"})   # 관련 있음 → 답변 / 없음 → 웹 검색
builder.add_edge("web_search", "generate")             # 웹 검색 → 답변
builder.add_edge("generate", "is_grounded")            # 답변 → 근거 확인
builder.add_conditional_edges("is_grounded", decide_after_grounded,   # 근거 확인 후 갈림길 (통과 / 다시 쓰기 / 포기)
                              {"grounded": "is_useful", "retry": "regenerate", "give_up": "fallback"})
builder.add_edge("regenerate", "is_grounded")          # 다시 쓴 답변도 근거 확인을 다시 받음 (루프)
builder.add_conditional_edges("is_useful", decide_after_useful,   # 유용성 확인 후 갈림길 (종료 / fallback)
                              {"useful": END, "not_useful": "fallback"})
builder.add_edge("fallback", END)                      # 안내 문구 → 끝
crag_graph = builder.compile()                         # 완성

# 그래프 연결 구조 확인 (__start__ = START, __end__ = END)
drawn = crag_graph.get_graph()                                              # 그래프 구조(노드·엣지) 정보 꺼내기
node_order = list(drawn.nodes)                                       # 노드를 등록한 순서
for edge in sorted(drawn.edges, key=lambda e: node_order.index(e.source)):   # 출발 노드 순서대로 정렬해서 하나씩
    label = ""                                                       # 기본값: 조건 표시 없음
    if edge.conditional:                                             # 조건부 연결이면 어떤 조건인지 표시
        label = f"   (조건부: {edge.data})" if edge.data else "   (조건부)"
    print(edge.source, "→", edge.target, label)

__start__ → retrieve 
retrieve → grade_documents 
grade_documents → generate    (조건부)
grade_documents → web_search    (조건부)
web_search → generate 
generate → is_grounded 
is_grounded → fallback    (조건부: give_up)
is_grounded → is_useful    (조건부: grounded)
is_grounded → regenerate    (조건부: retry)
regenerate → is_grounded 
is_useful → __end__    (조건부: useful)
is_useful → fallback    (조건부: not_useful)
fallback → __end__ 


## **(4) 실행**

In [ ]:
# 🔑 [Colab에서 실행] CRAG + Self-RAG 챗봇 실행 — 어떤 노드를 거쳤는지(평가 결과 포함)도 출력합니다 (API 키 + 벡터DB 필요)
def run_with_trace(graph, question):
    """그래프를 실행하면서 거쳐 간 노드와 평가 결과를 출력하고, 최종 State를 돌려줍니다."""
    final_state = None                                            # 최종 State를 담을 변수
    inputs = {"messages": [HumanMessage(content=question)], "retry_count": 0}   # 처음 State
    for mode, chunk in graph.stream(inputs, stream_mode=["updates", "values"]):
        if mode == "updates":                                     # updates = 방금 실행된 노드가 바꾼 칸
            for node_name, update in chunk.items():
                info = []                                         # 출력할 정보 모음
                if "documents" in update:                         # 문서 칸이 바뀌었으면 문서 개수 표시
                    info.append(f"문서 {len(update['documents'])}개")
                for key in ("grade", "grounded", "useful", "retry_count"):   # 평가 결과 표시
                    if key in update:
                        info.append(f"{key}={update[key]}")
                print(f"  ▶ {node_name}  {' | '.join(info)}")
        else:                                                     # values = 그 시점의 전체 State
            final_state = chunk                                   # 마지막으로 받은 값이 최종 State
    return final_state


crag_result = run_with_trace(crag_graph, "고혈압 환자가 주의해야 할 음식은?")   # 실행 경로를 출력하며 실행
print("\n[최종 답변]\n" + crag_result["answer"])

# **4. 검증 없음 vs 있음 비교**

* 아래 3개 질문으로 **기본 RAG 챗봇**(검증 없음)과 **CRAG+Self-RAG 챗봇**(검증 있음)의 답변을 비교합니다.
* 질문 구성
    * DB에 관련 정보가 있는 질문
    * DB와 무관한 질문 (CRAG의 웹 검색 보완이 동작해야 하는 경우)
    * 문서에 없는 세부사항까지 묻는, 답변이 과장되기 쉬운 질문 (Self-RAG의 근거 확인이 동작해야 하는 경우)

In [12]:
test_questions = [
    "고혈압 환자가 주의해야 할 음식은?",
    "오늘 원달러 환율은 얼마야?",
    "당뇨병 환자가 먹으면 안되는 특정 브랜드 라면 이름을 알려줘",
]

In [ ]:
# 🔑 [Colab에서 실행] ① 기본 RAG 챗봇(검증 없음)으로 3개 질문 실행 (API 키 + 벡터DB 필요)
basic_answers = {}   # 질문 → 답변 (아래 셀에서 나란히 비교할 때 사용)
for q in test_questions:
    state = basic_graph.invoke({"messages": [HumanMessage(content=q)]})   # 기본 RAG 실행
    basic_answers[q] = state["answer"]                                   # 답변 저장
    print("=" * 90)
    print("❓", q)
    print("\n[기본 RAG — 검증 없음]\n" + state["answer"])

In [ ]:
# 🔑 [Colab에서 실행] ② CRAG + Self-RAG 챗봇(검증 있음)으로 같은 질문 실행 → 실행 경로와 함께 나란히 비교
crag_answers = {}   # 질문 → 검증 있는 답변
for q in test_questions:
    print("=" * 90)
    print("❓", q)
    print("[CRAG + Self-RAG 실행 경로]")
    state = run_with_trace(crag_graph, q)              # 거쳐 간 노드와 평가 결과(grade/grounded/useful) 출력
    crag_answers[q] = state["answer"]                  # 최종 답변 저장
    print("\n[검증 있음 — 최종 답변]\n" + state["answer"])
    print("\n[비교: 검증 없음]\n" + basic_answers[q])

**질문**: 3개 질문 중 어떤 경우에 두 챗봇의 답변이 가장 크게 달랐나요? 검증이 없을 때 기본 RAG 챗봇이 보이는 문제(예: 무관한 문서로 억지 답변, 근거 없는 내용 포함 등)를 구체적으로 짚어보고, CRAG/Self-RAG의 어떤 단계가 그 문제를 해결했는지 설명해보세요.

**답변 (조태희)**

> ✍️ 아래는 각 검증 단계의 역할에서 **예상되는 결과**를 정리한 것입니다. Colab에서 4절을 실행한 뒤 '실행 경로'(어떤 노드를 거쳤는지)와 두 답변을 비교해서, 실제로 가장 크게 달랐던 질문과 그 근거를 덧붙여 주세요.

**예상: 두 번째 질문(원달러 환율)에서 가장 크게 달라질 것으로 봅니다.** DB에는 만성질환 정보만 있어서 기본 RAG는 환율과 무관한 고혈압·당뇨 문서를 참고 문서로 받습니다. 검증이 없으니 "문서에 환율 정보가 없다"고 끝내거나, 무관한 문서를 억지로 엮어 답하거나, 모델이 기억하는 예전 환율을 근거 없이 말할 위험이 있습니다. 반면 검증 있는 챗봇은 CRAG의 `grade_documents`가 세 문서를 모두 'no'로 판정하면 `web_search`로 넘어가, 웹 검색 결과를 근거로 답하게 됩니다.

세 번째 질문(특정 브랜드 라면)은 DB에 당뇨 식단 정보는 있지만 브랜드 이름은 없는 경우입니다. 기본 RAG는 "구체적으로 답하라"는 지침 때문에 문서에 없는 제품명을 지어내거나 과장할 위험이 있습니다. 검증 있는 챗봇에서는 Self-RAG의 `is_grounded`가 문서에 없는 내용을 'no'로 잡아내면 `regenerate`가 문서에 있는 내용만으로 다시 쓰고, 그래도 근거가 부족하거나 `is_useful`이 'no'면 `fallback`이 "근거를 찾지 못했다"고 안전하게 안내합니다. (CRAG가 이 질문의 DB 문서를 '관련 없음'으로 보면 웹 검색으로 넘어갈 수도 있으니, 실제 경로를 꼭 확인해 보세요.)

첫 번째 질문(고혈압 음식)은 DB에 관련 정보가 있어서 두 챗봇의 답이 비슷할 가능성이 높고, 차이는 검증 있는 쪽의 답변이 근거 확인과 유용성 확인을 통과했다는 점입니다.

**정리:** 검증 없는 RAG는 "검색된 문서가 질문에 맞는지", "답이 문서에 근거하는지"를 확인하지 않아서 무관한 문서로 억지 답변을 하거나 근거 없는 내용을 그럴듯하게 말할 수 있습니다. CRAG(문서 관련성 평가 + 웹 검색 보완)는 **잘못된 재료**를, Self-RAG(근거 확인 + 유용성 확인)는 **잘못된 답변**을 걸러 주는 역할을 합니다.